# Data Module Sanity Notebook

This notebook contains interactive checks for:

1. Dataset-level sanity
2. Wrapper-level sanity
3. Loader-level sanity

Run top-to-bottom.

## 0) Setup

In [1]:
import sys
sys.path.append('..')

from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import torch

from src.config import load_config
from src.data.datasets import build_dataset
from src.data.wrappers import LabelRouterDataset, IdFilteredDataset, TwoViewDataset
from src.data.utils import (
    build_pretrain_loaders,
    build_target_adapt_base,
    build_round_select_pool_loader,
    build_round_train_loaders,
    build_static_eval_loaders,
)

CFG_SOURCE = Path('../configs/train_source_office_home.yaml')
CFG_FT = Path('../configs/finetune_target_office_home.yaml')

cfg_source = load_config(CFG_SOURCE)
cfg_ft = load_config(CFG_FT)

print('source cfg:', CFG_SOURCE)
print('finetune cfg:', CFG_FT)
print('dataset:', cfg_ft.data.dataset_name)
print('source domain:', cfg_ft.data.source_domain)
print('target domain:', cfg_ft.data.target_domain)

source cfg: ../configs/train_source_office_home.yaml
finetune cfg: ../configs/finetune_target_office_home.yaml
dataset: office_home
source domain: Art
target domain: Clipart


## 1) Dataset-Level Sanity

Checks:

- each required split can be built
- each split is non-empty
- `sample_id` is deterministic and unique
- label ranges look reasonable

In [5]:
splits = ['source_train', 'source_val', 'target_adapt', 'target_test']
datasets = {}

for split in splits:
    ds = build_dataset(cfg_ft, split=split, transform=None)
    datasets[split] = ds
    print(f'{split:12s} size={len(ds)}')
    assert len(ds) > 0, f'{split} is empty'

print('All splits built and non-empty.')

source_train size=2209
source_val   size=218
target_adapt size=4365
target_test  size=4365
All splits built and non-empty.


In [6]:
for split, ds in datasets.items():
    # determinism
    assert ds[0]['sample_id'] == ds[0]['sample_id'], f'{split}: sample_id not deterministic'

    # uniqueness
    ids = [ds[i]['sample_id'] for i in range(len(ds))]
    assert len(ids) == len(set(ids)), f'{split}: sample_id not unique'

    labels = [ds[i]['label'] for i in range(len(ds))]
    c = Counter(labels)
    print(f'[{split}] classes={len(c)} min={min(c)} max={max(c)} top5={c.most_common(5)}')

print('Dataset-level checks passed.')

[source_train] classes=65 min=0 max=64 top5=[(5, 90), (22, 81), (27, 72), (9, 69), (4, 68)]
[source_val] classes=65 min=0 max=64 top5=[(5, 9), (22, 9), (9, 7), (4, 7), (27, 7)]
[target_adapt] classes=65 min=0 max=64 top5=[(4, 99), (5, 99), (9, 99), (10, 99), (12, 99)]
[target_test] classes=65 min=0 max=64 top5=[(4, 99), (5, 99), (9, 99), (10, 99), (12, 99)]
Dataset-level checks passed.


## 2) Wrapper-Level Sanity

Checks:

- `LabelRouterDataset`: queried->GT, pseudo->pseudo label, else `-1`
- `IdFilteredDataset`: labeled/pseudo/pool are disjoint and cover full target set
- `TwoViewDataset`: output contract

In [2]:
base = build_dataset(cfg_ft, split='target_adapt', transform=None)

# smoke test
queried_ids = {base[i]['sample_id'] for i in range(10)}
pseudo_store = {base[i]['sample_id']: (base[i]['label'] + 1) % cfg_ft.data.num_classes for i in range(10, 20)}

routed = LabelRouterDataset(base, queried_ids=queried_ids, pseudo_store=pseudo_store, unlabeled_label=-1)

# queried -> GT
for i in range(10):
    item = routed[i]
    assert item['label'] == base[i]['label']

# pseudo -> pseudo label
for i in range(10, 20):
    item = routed[i]
    assert item['label'] == pseudo_store[item['sample_id']]

# others -> -1
for i in range(20, 30):
    item = routed[i]
    assert item['sample_id'] not in queried_ids
    assert item['sample_id'] not in pseudo_store
    assert item['label'] == -1

print('LabelRouterDataset checks passed.')

LabelRouterDataset checks passed.


In [3]:
labeled_ds = IdFilteredDataset(routed, mode='labeled')
pseudo_ds = IdFilteredDataset(routed, mode='pseudo')
pool_ds = IdFilteredDataset(routed, mode='pool')

L = {labeled_ds[i]['sample_id'] for i in range(len(labeled_ds))}
P = {pseudo_ds[i]['sample_id'] for i in range(len(pseudo_ds))}
U = {pool_ds[i]['sample_id'] for i in range(len(pool_ds))}
ALL = {base[i]['sample_id'] for i in range(len(base))}

assert L.isdisjoint(P)
assert L.isdisjoint(U)
assert P.isdisjoint(U)
assert L | P | U == ALL

print('IdFilteredDataset partition checks passed.')
print('sizes:', len(L), len(P), len(U), 'total:', len(ALL))

IdFilteredDataset partition checks passed.
sizes: 10 10 4345 total: 4365


In [4]:
def weak_tf(_img):
    return torch.zeros(3, 16, 16)

def strong_tf(_img):
    return torch.ones(3, 16, 16)

two_view = TwoViewDataset(pool_ds, weak_tf=weak_tf, strong_tf=strong_tf)
item = two_view[0]

assert set(item.keys()) == {'x_w', 'x_s', 'label', 'sample_id'}
assert tuple(item['x_w'].shape) == (3, 16, 16)
assert tuple(item['x_s'].shape) == (3, 16, 16)
print('TwoViewDataset contract check passed.')

TwoViewDataset contract check passed.


In [6]:
from src.data.transforms import build_weak_transform, build_strong_transform
weak_tf = build_weak_transform(cfg_ft)
strong_tf = build_strong_transform(cfg_ft)
two_view = TwoViewDataset(pool_ds, weak_tf=weak_tf, strong_tf=strong_tf)

## 3) Loader-Level Sanity

Checks:

- `build_pretrain_loaders` returns source loaders with expected keys
- `build_round_select_pool_loader` builds selection pool (`target_adapt \ queried`)
- `build_round_train_loaders` builds round-dependent train loaders
- `build_static_eval_loaders` returns static eval loaders

In [4]:
pre = build_pretrain_loaders(cfg_source)
print('pretrain loader keys:', list(pre.keys()))
assert 'source_train' in pre and 'source_val' in pre
assert len(pre['source_train'].dataset) > 0
assert len(pre['source_val'].dataset) > 0

b = next(iter(pre['source_train']))
assert {'image', 'label', 'sample_id'}.issubset(set(b.keys()))
print('Pretrain loader checks passed.')

pretrain loader keys: ['source_train', 'source_val']
Pretrain loader checks passed.


In [5]:
@dataclass
class RoundStateLite:
    queried_ids: set
    pseudo_store: dict

target_adapt_base = build_target_adapt_base(cfg_ft)
state_a = RoundStateLite(queried_ids=set(), pseudo_store={})
select_loader = build_round_select_pool_loader(cfg_ft, target_adapt_base, state_a)
bsel = next(iter(select_loader))
print('select keys:', sorted(bsel.keys()))
assert set(bsel.keys()) == {'image', 'label', 'sample_id'}

sid0 = bsel['sample_id'][0]
state_b = RoundStateLite(queried_ids={sid0}, pseudo_store={})
train_loaders_b = build_round_train_loaders(cfg_ft, target_adapt_base, state_b)
print('train keys (no pseudo):', sorted(train_loaders_b.keys()))
assert 'target_adapt_labeled' in train_loaders_b
assert 'target_adapt_pseudo' not in train_loaders_b

sid1 = bsel['sample_id'][1]
state_c = RoundStateLite(queried_ids={sid0}, pseudo_store={sid1: 0})
train_loaders_c = build_round_train_loaders(cfg_ft, target_adapt_base, state_c)
print('train keys (with pseudo):', sorted(train_loaders_c.keys()))
assert 'target_adapt_pseudo' in train_loaders_c

eval_loaders = build_static_eval_loaders(cfg_ft)
print('static eval keys:', sorted(eval_loaders.keys()))
assert 'target_test' in eval_loaders
print('Loader builder checks passed.')

select keys: ['image', 'label', 'sample_id']
train keys (no pseudo): ['target_adapt_labeled']
train keys (with pseudo): ['target_adapt_labeled', 'target_adapt_pseudo']
static eval keys: ['source_val', 'target_test']
Loader builder checks passed.


In [6]:
bl = next(iter(train_loaders_b['target_adapt_labeled']))
assert set(bl.keys()) == {'x_w', 'x_s', 'label', 'sample_id'}
print('Round train loader batch contract check passed.')

print('All data module sanity checks passed.')

Round train loader batch contract check passed.
All data module sanity checks passed.
